In [ ]:
from __future__ import annotations

import os
import csv
import pickle
import re
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
from Bio import SeqIO
from scipy.linalg import expm

In [ ]:
# ============================================================
# Project / paths
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()
PROJECT_ROOT = REPOSITORY_ROOT / "AA"
DATA_ROOT = PROJECT_ROOT / "data"

SIM_DIR = DATA_ROOT / "sim"
EVO_DISTANCE_DIR = DATA_ROOT / "evo_distances"

MANIFEST_PATH = DATA_ROOT / "manifests" / "simulation_manifest.csv"

MODEL_DIR = PROJECT_ROOT / "models"
WAG_DAT_PATH = MODEL_DIR / "wag.dat"
WAG_Q_CSV = MODEL_DIR / "WAG_Q.csv"


# ============================================================
# Evolutionary-distance parameters
# ============================================================

STANDARD_AA = "ARNDCQEGHILKMFPSTWYV"

EPS = 1e-12

# Poisson20:
# d = -(19/20) log(1 - (20/19)p)
POISSON20_MAX_P = 19.0 / 20.0

# WAG ML distance grid
WAG_T_MAX = 5.0
WAG_GRID_SIZE = 501

# False: use existing WAG_Q.csv if present
# True: regenerate WAG_Q.csv from wag.dat
OVERWRITE_WAG_Q = False

# False: keep existing evolutionary-distance outputs
# True: regenerate existing outputs
OVERWRITE = False

In [ ]:
@dataclass
class EvoDistanceRow:
    tag: str
    fasta_path: str
    n_sequences: int

    pickle_path: str
    npz_path: str

    status: str
    message: str


def ensure_directories() -> None:
    EVO_DISTANCE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    MODEL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


def read_numbers_from_wag_dat(
    wag_dat_path: Path,
) -> list[float]:
    text = wag_dat_path.read_text(
        encoding="utf-8"
    )

    numbers = [
        float(x)
        for x in re.findall(
            r"[-+]?\d*\.\d+(?:[Ee][-+]?\d+)?|[-+]?\d+(?:[Ee][-+]?\d+)?",
            text,
        )
    ]

    return numbers


def create_wag_q_csv_from_dat(
    wag_dat_path: Path,
    output_path: Path,
) -> None:
    aa_order = list(STANDARD_AA)

    n_aa = len(aa_order)
    n_exchangeabilities = n_aa * (n_aa - 1) // 2

    numbers = read_numbers_from_wag_dat(
        wag_dat_path
    )

    required = n_exchangeabilities + n_aa

    if len(numbers) < required:
        raise ValueError(
            "WAG source file does not contain enough numeric values. "
            f"expected at least {required}, got {len(numbers)}"
        )

    exchangeabilities = numbers[:n_exchangeabilities]

    pi = np.array(
        numbers[n_exchangeabilities:required],
        dtype=float,
    )

    pi = pi / pi.sum()

    S = np.zeros(
        (n_aa, n_aa),
        dtype=float,
    )

    index = 0

    for i in range(1, n_aa):
        for j in range(i):
            value = exchangeabilities[index]
            S[i, j] = value
            S[j, i] = value
            index += 1

    Q = np.zeros(
        (n_aa, n_aa),
        dtype=float,
    )

    for i in range(n_aa):
        for j in range(n_aa):
            if i != j:
                Q[i, j] = S[i, j] * pi[j]

    np.fill_diagonal(
        Q,
        -Q.sum(axis=1),
    )

    mean_rate = -np.sum(
        pi * np.diag(Q)
    )

    Q = Q / mean_rate

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df_Q = pd.DataFrame(
        Q,
        index=aa_order,
        columns=aa_order,
    )

    df_Q.to_csv(
        output_path,
    )


def ensure_wag_q_matrix() -> None:
    if WAG_Q_CSV.exists() and not OVERWRITE_WAG_Q:
        return

    if not WAG_DAT_PATH.exists():
        raise FileNotFoundError(
            f"WAG source file was not found: {WAG_DAT_PATH}"
        )

    create_wag_q_csv_from_dat(
        wag_dat_path=WAG_DAT_PATH,
        output_path=WAG_Q_CSV,
    )


def check_inputs() -> None:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Simulation manifest was not found: {MANIFEST_PATH}"
        )

    ensure_wag_q_matrix()

    if not WAG_Q_CSV.exists():
        raise FileNotFoundError(
            f"WAG Q matrix was not found: {WAG_Q_CSV}"
        )


def read_simulation_manifest() -> list[dict]:
    rows: list[dict] = []

    with MANIFEST_PATH.open(newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("status", "") in {"ok", "skipped_existing"}:
                rows.append(row)

    return rows


def resolve_fasta_path(
    row: dict,
) -> Path:
    manifest_path = Path(row["fasta_path"]).expanduser()

    if manifest_path.is_absolute():
        candidates = [manifest_path]
    else:
        candidates = [PROJECT_ROOT / manifest_path]

    # Backward-compatible fallback for manifests that only encode the tag.
    candidates.append(
        SIM_DIR / f"{row['tag']}.fa"
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"FASTA file was not found for tag={row['tag']}. "
        f"Checked: {checked}"
    )


def read_fasta_records(
    fasta_path: Path,
) -> tuple[list[str], list[str]]:
    records = list(
        SeqIO.parse(
            str(fasta_path),
            "fasta",
        )
    )

    ids = [
        record.id
        for record in records
    ]

    sequences = [
        str(record.seq).upper()
        for record in records
    ]

    return ids, sequences


def encode_aa_sequences(
    sequences: list[str],
) -> np.ndarray:
    aa_to_index = {
        aa: index
        for index, aa in enumerate(STANDARD_AA)
    }

    n = len(sequences)
    length = len(sequences[0])

    encoded = np.full(
        (n, length),
        fill_value=-1,
        dtype=int,
    )

    for i, sequence in enumerate(sequences):
        if len(sequence) != length:
            raise ValueError(
                "All sequences must have the same length."
            )

        for j, aa in enumerate(sequence):
            encoded[i, j] = aa_to_index.get(
                aa,
                -1,
            )

    return encoded


def pairwise_p_distance_aa(
    encoded: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    n = encoded.shape[0]

    p_distance = np.zeros(
        (n, n),
        dtype=float,
    )

    valid_sites = np.zeros(
        (n, n),
        dtype=int,
    )

    for i in range(n):
        for j in range(i + 1, n):
            valid = (
                (encoded[i] >= 0)
                & (encoded[j] >= 0)
            )

            n_valid = int(
                valid.sum()
            )

            valid_sites[i, j] = n_valid
            valid_sites[j, i] = n_valid

            if n_valid == 0:
                p = np.nan

            else:
                mismatches = (
                    encoded[i, valid]
                    != encoded[j, valid]
                )

                p = float(
                    mismatches.mean()
                )

            p_distance[i, j] = p
            p_distance[j, i] = p

    np.fill_diagonal(
        p_distance,
        0.0,
    )

    return p_distance, valid_sites


def poisson20_corrected_distance(
    p_distance: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    p = np.asarray(
        p_distance,
        dtype=float,
    )

    clip_threshold = POISSON20_MAX_P - EPS

    clipped_p = np.clip(
        p,
        0.0,
        clip_threshold,
    )

    clipped_mask = (
        p >= POISSON20_MAX_P
    )

    distance = -(
        19.0
        / 20.0
    ) * np.log(
        1.0
        - (
            20.0
            / 19.0
        )
        * clipped_p
    )

    finite = distance[np.isfinite(distance)]

    fallback = (
        float(finite.max())
        if finite.size > 0
        else 0.0
    )

    distance = np.nan_to_num(
        distance,
        nan=fallback,
        posinf=fallback,
        neginf=0.0,
    )

    distance = 0.5 * (
        distance
        + distance.T
    )

    np.fill_diagonal(
        distance,
        0.0,
    )

    return distance, clipped_mask


def load_wag_q_matrix(
    path: Path,
) -> np.ndarray:
    df = pd.read_csv(
        path,
        index_col=0,
    )

    missing_rows = set(STANDARD_AA) - set(df.index)
    missing_cols = set(STANDARD_AA) - set(df.columns)

    if missing_rows or missing_cols:
        raise ValueError(
            "WAG Q CSV must contain all standard amino acids. "
            f"missing_rows={missing_rows}, "
            f"missing_cols={missing_cols}"
        )

    df = df.loc[
        list(STANDARD_AA),
        list(STANDARD_AA),
    ]

    q_matrix = df.to_numpy(
        dtype=float,
    )

    return q_matrix


def precompute_wag_log_probabilities(
    q_matrix: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    t_grid = np.linspace(
        0.0,
        WAG_T_MAX,
        WAG_GRID_SIZE,
    )

    log_probabilities = np.empty(
        (
            len(t_grid),
            len(STANDARD_AA),
            len(STANDARD_AA),
        ),
        dtype=float,
    )

    for index, t in enumerate(t_grid):
        p_matrix = expm(
            q_matrix
            * t
        )

        p_matrix = np.maximum(
            p_matrix,
            EPS,
        )

        log_probabilities[index] = np.log(
            p_matrix
        )

    return t_grid, log_probabilities


def pair_count_matrix(
    encoded_i: np.ndarray,
    encoded_j: np.ndarray,
) -> np.ndarray:
    valid = (
        (encoded_i >= 0)
        & (encoded_j >= 0)
    )

    counts = np.zeros(
        (
            len(STANDARD_AA),
            len(STANDARD_AA),
        ),
        dtype=float,
    )

    if not np.any(valid):
        return counts

    np.add.at(
        counts,
        (
            encoded_i[valid],
            encoded_j[valid],
        ),
        1.0,
    )

    return counts


def wag_ml_distance_matrix(
    encoded: np.ndarray,
    t_grid: np.ndarray,
    log_probabilities: np.ndarray,
) -> np.ndarray:
    n = encoded.shape[0]

    pairs: list[tuple[int, int]] = []
    count_vectors: list[np.ndarray] = []

    for i in range(n):
        for j in range(i + 1, n):
            counts = pair_count_matrix(
                encoded[i],
                encoded[j],
            )

            pairs.append(
                (
                    i,
                    j,
                )
            )

            count_vectors.append(
                counts.reshape(-1)
            )

    if not pairs:
        return np.zeros(
            (n, n),
            dtype=float,
        )

    count_matrix = np.vstack(
        count_vectors
    )

    log_prob_matrix = log_probabilities.reshape(
        len(t_grid),
        -1,
    )

    log_likelihoods = (
        count_matrix
        @ log_prob_matrix.T
    )

    valid_pair = (
        count_matrix.sum(axis=1)
        > 0
    )

    best_indices = np.argmax(
        log_likelihoods,
        axis=1,
    )

    best_t = t_grid[
        best_indices
    ].astype(float)

    best_t[~valid_pair] = WAG_T_MAX

    distance = np.zeros(
        (n, n),
        dtype=float,
    )

    for value, (i, j) in zip(best_t, pairs):
        distance[i, j] = value
        distance[j, i] = value

    np.fill_diagonal(
        distance,
        0.0,
    )

    return distance


def output_is_complete(
    pickle_path: Path,
    npz_path: Path,
) -> bool:
    if not pickle_path.exists():
        return False

    if not npz_path.exists():
        return False

    try:
        with pickle_path.open("rb") as f:
            payload = pickle.load(f)

    except Exception:
        return False

    required_keys = {
        "p_distance",
        "valid_sites",
        "poisson20_distance",
        "wag_distance",
    }

    return required_keys.issubset(
        payload.keys()
    )


def save_pickle(
    obj,
    output_path: Path,
) -> None:
    with output_path.open("wb") as f:
        pickle.dump(
            obj,
            f,
        )


def save_npz(
    output_path: Path,
    **arrays,
) -> None:
    np.savez_compressed(
        output_path,
        **arrays,
    )


def save_evo_manifest(
    rows: list[EvoDistanceRow],
    output_path: Path,
) -> None:
    fieldnames = list(
        EvoDistanceRow.__annotations__.keys()
    )
    path_fields = {
        "fasta_path",
        "pickle_path",
        "npz_path",
    }
    root = PROJECT_ROOT.resolve()

    with output_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:
            record = asdict(row)

            for field in path_fields:
                value = record.get(field)

                if not value:
                    continue

                path = Path(value).expanduser().resolve()

                try:
                    record[field] = (
                        path.relative_to(root).as_posix()
                    )
                except ValueError as error:
                    raise ValueError(
                        f"{field} is outside project root: {path}"
                    ) from error

            writer.writerow(record)


def process_one_dataset(
    tag: str,
    fasta_path: Path,
    t_grid: np.ndarray,
    wag_log_probabilities: np.ndarray,
) -> EvoDistanceRow:
    pickle_path = (
        EVO_DISTANCE_DIR
        / f"{tag}_evo_distances.pkl"
    )

    npz_path = (
        EVO_DISTANCE_DIR
        / f"{tag}_evo_distances.npz"
    )

    row = EvoDistanceRow(
        tag=tag,
        fasta_path=str(fasta_path),
        n_sequences=0,
        pickle_path=str(pickle_path),
        npz_path=str(npz_path),
        status="planned",
        message="",
    )

    # Read FASTA before the skip check so that n_sequences is recorded
    # even when the output already exists.
    try:
        ids, sequences = read_fasta_records(
            fasta_path
        )

    except Exception as error:
        return replace(
            row,
            status="failed",
            message=str(error)[:2000],
        )

    row = replace(
        row,
        n_sequences=len(ids),
    )

    if (
        output_is_complete(
            pickle_path,
            npz_path,
        )
        and not OVERWRITE
    ):
        return replace(
            row,
            status="ok",
            message="skipped_existing",
        )

    try:
        encoded = encode_aa_sequences(
            sequences
        )

        p_distance, valid_sites = pairwise_p_distance_aa(
            encoded
        )

        poisson20_distance, poisson20_clipped = poisson20_corrected_distance(
            p_distance
        )

        wag_distance = wag_ml_distance_matrix(
            encoded=encoded,
            t_grid=t_grid,
            log_probabilities=wag_log_probabilities,
        )

        payload = {
            "tag": tag,
            "ids": ids,
            "p_distance": p_distance,
            "valid_sites": valid_sites,
            "poisson20_distance": poisson20_distance,
            "poisson20_clipped": poisson20_clipped,
            "wag_distance": wag_distance,
            "wag_q_csv": str(WAG_Q_CSV),
            "wag_t_max": WAG_T_MAX,
            "wag_grid_size": WAG_GRID_SIZE,
        }

        save_pickle(
            payload,
            pickle_path,
        )

        save_npz(
            npz_path,
            p_distance=p_distance,
            valid_sites=valid_sites,
            poisson20_distance=poisson20_distance,
            poisson20_clipped=poisson20_clipped,
            wag_distance=wag_distance,
        )

    except Exception as error:
        return replace(
            row,
            status="failed",
            message=str(error)[:2000],
        )

    return replace(
        row,
        status="ok",
        message="",
    )

In [ ]:
ensure_directories()
check_inputs()

q_matrix = load_wag_q_matrix(
    WAG_Q_CSV
)

t_grid, wag_log_probabilities = precompute_wag_log_probabilities(
    q_matrix
)

simulation_rows = read_simulation_manifest()

evo_rows: list[EvoDistanceRow] = []

for index, sim_row in enumerate(simulation_rows, start=1):
    tag = sim_row["tag"]
    fasta_path = resolve_fasta_path(
        sim_row
    )

    print(
        f"[{index:04d}/{len(simulation_rows):04d}] "
        f"{tag}"
    )

    evo_rows.append(
        process_one_dataset(
            tag=tag,
            fasta_path=fasta_path,
            t_grid=t_grid,
            wag_log_probabilities=wag_log_probabilities,
        )
    )

evo_manifest_path = (
    EVO_DISTANCE_DIR
    / "evo_distance_manifest.csv"
)

save_evo_manifest(
    evo_rows,
    evo_manifest_path,
)

status_counts = Counter(
    row.status
    for row in evo_rows
)

message_counts = Counter(
    row.message
    for row in evo_rows
)

print()
print("Evolutionary distance construction completed.")

for status, count in sorted(status_counts.items()):
    print(f"{status}: {count}")

print()
print("Messages:")

for message, count in sorted(message_counts.items()):
    label = message if message else "generated"
    print(f"{label}: {count}")

print()
print(f"Evolutionary distance manifest: {evo_manifest_path}")